# Exercise 1: Data scarcity analysis

*Machine Translation · version 0.1.0*

> Using the chapter’s TIL data table as a reference, compute the ratio between English–Turkish and English–Uyghur sentence pair counts. Choose two differently resourced pairs and design a learning-curve experiment. Specify domain, metric and acceptance criteria; do not assume a universal sentence-count threshold.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" torch transformers sentencepiece sacrebleu requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`). NLLB-200 distilled 600M is loaded from `NLLB_PATH` (a local copy) or from the Hugging Face Hub at the revision pinned in the book; it runs in float32 on the CPU, about 1.7 s per sentence on four cores. Translations are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes where it stopped. Test data come from the commit-pinned TIL release (X-WMT test sets, MIT licence); the X-WMT files are tokenised, and the code removes the spaces around punctuation before scoring (a heuristic, see `detok`). Scores use SacreBLEU with its default settings: BLEU with the `13a` tokeniser, chrF, and chrF++ (`word_order=2`).

The counts are the "Training size" column of the release README (`til_corpus/README.md` at commit `7d988cc`), the same source as the chapter's table.

In [ ]:
import collections, hashlib, os, re, statistics, time
from pathlib import Path
import requests

# Commit-pinned TIL release files (X-WMT test sets and the corpus README); MIT licence.
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation) so references match raw MT output."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)       # Sialkot ’ taki -> Sialkot’taki
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt(src, tgt, n=None):
    """The first n sentence pairs of the X-WMT test set for src-tgt (X-WMT language codes)."""
    s = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{src}").read_text(encoding="utf-8").splitlines()
    t = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{tgt}").read_text(encoding="utf-8").splitlines()
    assert len(s) == len(t), (src, tgt)
    pairs = [(detok(a), detok(b)) for a, b in zip(s, t) if a.strip() and b.strip()]
    return pairs[:n] if n else pairs

NLLB_CODE = {"az": "azj_Latn", "ba": "bak_Cyrl", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "tr": "tur_Latn",
             "uz": "uzn_Latn", "en": "eng_Latn", "ru": "rus_Cyrl"}
NLLB = {}

def nllb():
    """NLLB-200 distilled 600M: from NLLB_PATH (a local copy) or the pinned Hub revision; float32, CPU."""
    if not NLLB:
        import torch
        from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
        path = os.environ.get("NLLB_PATH", "facebook/nllb-200-distilled-600M")
        rev = None if Path(path).exists() else "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
        NLLB["tok"] = NllbTokenizer.from_pretrained(path, revision=rev)
        NLLB["model"] = AutoModelForSeq2SeqLM.from_pretrained(path, revision=rev, torch_dtype=torch.float32).eval()
        NLLB["torch"] = torch
        torch.set_num_threads(os.cpu_count() or 1)
    return NLLB["tok"], NLLB["model"]

CACHE_FILE = CACHE / "nllb-translations.jsonl"       # translations are cached, so a run can resume
LAST_SECONDS = 0.0

def _cached():
    import json
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[tuple(r["key"])] = (r["out"], r["capped"], r["sec"])
    return out

def translate(sents, src, tgt, beams=1, batch=8, max_new=256):
    """Translate with explicit source and target language tokens (greedy unless beams > 1).
    Returns (outputs, number of outputs that reached the generation ceiling without an
    end-of-sequence token). Results are cached per sentence in CACHE_FILE, with the generation
    time of their batch divided by its size; LAST_SECONDS holds the total for the last call."""
    global LAST_SECONDS
    import json
    cache = _cached()
    key = lambda s: [src, tgt, beams, max_new, s]
    todo = [s for s in dict.fromkeys(sents) if tuple(key(s)) not in cache]
    if todo:
        tok, model = nllb()
        tok.src_lang = NLLB_CODE[src]
        CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        with NLLB["torch"].inference_mode(), CACHE_FILE.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), batch):
                chunk = todo[i:i + batch]
                t0 = time.time()
                enc = tok(chunk, return_tensors="pt", padding=True)
                gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                                     num_beams=beams, do_sample=False, max_new_tokens=max_new)
                sec = (time.time() - t0) / len(chunk)
                for s, row, text in zip(chunk, gen, tok.batch_decode(gen, skip_special_tokens=True)):
                    capped = int(row[-1] != tok.eos_token_id and row[-1] != tok.pad_token_id)
                    cache[tuple(key(s))] = (text, capped, sec)
                    f.write(json.dumps({"key": key(s), "out": text, "capped": capped, "sec": sec}, ensure_ascii=False) + "\n")
                f.flush()
    res = [cache[tuple(key(s))] for s in sents]
    LAST_SECONDS = sum(r[2] for r in res)
    return [r[0] for r in res], sum(r[1] for r in res)

def metrics(hyps, refs):
    """Corpus BLEU (13a), chrF and chrF++ with SacreBLEU, rounded, plus the chrF++ signature."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu, chrf, chrfpp = BLEU(tokenize="13a"), CHRF(), CHRF(word_order=2)
    r = {"BLEU": bleu.corpus_score(hyps, [refs]).score, "chrF": chrf.corpus_score(hyps, [refs]).score,
         "chrF++": chrfpp.corpus_score(hyps, [refs]).score}
    return {k: round(v, 2) for k, v in r.items()}, str(chrfpp.get_signature())

In [ ]:
readme = fetch(TIL + "til_corpus/README.md").read_text(encoding="utf-8")
sizes = {}
for m in re.finditer(r"\|\s*\d+\s*\|\s*([^|]+?)\s*\|\s*([^|]+?)\s*\|\s*(\w+)\s*\|\s*(\w+)\s*\|\s*(\d+)\s*\|", readme):
    sizes[frozenset(m.group(3, 4))] = int(m.group(5))
n = lambda a, b: sizes[frozenset((a, b))]
print(f"{len(sizes)} language pairs with a training size in the release README")
print(f"English–Turkish {n('en', 'tr'):,} vs English–Uyghur {n('en', 'ug'):,}: ratio {n('en', 'tr') / n('en', 'ug'):.1f}")
v = sorted(sizes.values())
print(f"median pair size {statistics.median(v):,.0f}; pairs below 10,000: {sum(x < 10_000 for x in v)}; "
      f"below 1,000: {sum(x < 1_000 for x in v)}; above 1,000,000: {sum(x > 1_000_000 for x in v)}")
for a, b in [("kk", "ky"), ("en", "ky"), ("tr", "az"), ("uz", "ky")]:
    print(f"  {a}-{b}: {n(a, b):,}")

## Worked solution

Output of `experiments/ch10_exercises/run_ch10_exercises.py`:

```
202 language pairs with a training size in the release README
English–Turkish 35,879,592 vs English–Uyghur 96,898: ratio 370.3
median pair size 8,952; pairs below 10,000: 127; below 1,000: 2; above 1,000,000: 4
  kk-ky: 12,216
  en-ky: 312,644
  tr-az: 410,140
  uz-ky: 119,946
```

**The ratio.** English–Turkish has 35,879,592 training pairs and English–Uyghur 96,898: about **370 times** more. Across all 202 pairs of the release the median is only 8,952 pairs; 127 pairs have fewer than 10,000 and 4 have more than a million.

**A learning-curve design.** Take Kazakh–Kyrgyz (12,216 TIL pairs) and Uzbek–Kyrgyz (119,946), which share the target language.

- *Domain and test set:* the X-WMT Kazakh–Kyrgyz and Uzbek–Kyrgyz test sets (news, 500 sentences each), checked for overlap with the training files before use; the development set is a document-grouped slice of the training data, never X-WMT.
- *Sizes:* nested random subsets of 1,000, 3,000 and 10,000 pairs for both pairs, and 30,000 and the full 119,946 for Uzbek–Kyrgyz, with three seeds each; the same NLLB-200 starting point, decoding settings and training budget per update.
- *Metrics:* chrF++ as the primary metric, BLEU as a secondary one, with SacreBLEU signatures; a human error analysis of 50 sentences at the smallest and the largest size.
- *Acceptance criteria:* fixed before training, for example "chrF++ at least 3 points above the unadapted NLLB-200 baseline on the test set, and no increase in untranslated or wrong-language outputs in the manual sample". The size at which a pair meets them is the result; there is no universal threshold to compare it with.

## Your turn — extension

1. Plot the pair sizes of all Kyrgyz pairs in the release on a log scale.
2. Check how many of the X-WMT Kazakh–Kyrgyz test sentences occur in the TIL Kazakh–Kyrgyz training file.